# Regles de Gestion Metier - Data Quality BoursoBank

## Objectif
Implementer et verifier les 5 regles de gestion definies lors de l'analyse metier.
Chaque violation de regle = une anomalie a signaler et corriger.

## Regles implementees
- RG01 : Completude - ID client ne doit pas etre vide
- RG02 : Format - Age doit etre dans la liste de valeurs autorisees
- RG03 : Format - Genre doit etre M ou F uniquement
- RG04 : Coherence - Montant doit etre strictement superieur a 0
- RG05 : Unicite - Chaque transaction doit etre unique

In [1]:
import pandas as pd
import numpy as np

# Chargement du dataset
df = pd.read_csv("../data/bs140513_032310.csv")

print(f"Dataset charge : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

Dataset charge : 594,643 lignes x 10 colonnes


## RG01 - Completude : ID client ne doit pas etre vide
Toute transaction doit avoir un ID client.
Un ID client vide = transaction non rattachable a un client = risque reglementaire.

In [2]:
# RG01 - Completude
# On cherche les lignes ou customer est vide
# isnull() retourne True si la valeur est vide, False sinon

anomalies_rg01 = df[df["customer"].isnull()]

print(f"RG01 - Nombre d'anomalies : {len(anomalies_rg01)}")
print(f"Pourcentage : {len(anomalies_rg01) / len(df) * 100:.2f}%")

RG01 - Nombre d'anomalies : 0
Pourcentage : 0.00%


## RG02 - Format : Age doit etre dans la liste de valeurs autorisees
Les tranches d'age acceptees sont des valeurs definies.
Toute valeur hors de cette liste = anomalie de format.

In [3]:
# RG02 - Format Age
# On definit d'abord la liste des valeurs autorisees
# Puis on cherche les lignes qui ne sont pas dans cette liste

valeurs_autorisees_age = ["'1'", "'2'", "'3'", "'4'", "'5'", "'6'", "'U'"]

anomalies_rg02 = df[~df["age"].isin(valeurs_autorisees_age)]

print(f"RG02 - Valeurs autorisees : {valeurs_autorisees_age}")
print(f"RG02 - Nombre d'anomalies : {len(anomalies_rg02)}")
print(f"Valeurs anormales trouvees : {df['age'].unique()}")

RG02 - Valeurs autorisees : ["'1'", "'2'", "'3'", "'4'", "'5'", "'6'", "'U'"]
RG02 - Nombre d'anomalies : 2452
Valeurs anormales trouvees : <StringArray>
[''4'', ''2'', ''3'', ''5'', ''1'', ''6'', ''U'', ''0'']
Length: 8, dtype: str


In [4]:
# RG02 mise a jour - On separe les cas

# Valeurs techniquement valides
valeurs_format_ok = ["'1'", "'2'", "'3'", "'4'", "'5'", "'6'", "'U'"]

# Anomalies techniques - valeurs hors referentiel
anomalies_rg02_technique = df[~df["age"].isin(valeurs_format_ok)]

# Anomalies metier - mineurs et inconnus
anomalies_rg02_mineur = df[df["age"] == "'1'"]
anomalies_rg02_inconnu = df[df["age"] == "'U'"]

print(f"RG02 - Anomalies techniques (hors referentiel) : {len(anomalies_rg02_technique)}")
print(f"RG02 - Clients mineurs (tranche 1) : {len(anomalies_rg02_mineur)}")
print(f"RG02 - Age inconnu (U) : {len(anomalies_rg02_inconnu)}")

RG02 - Anomalies techniques (hors referentiel) : 2452
RG02 - Clients mineurs (tranche 1) : 58131
RG02 - Age inconnu (U) : 1178


## RG03 - Format : Genre doit etre M ou F uniquement
Valeurs acceptables : M (masculin) ou F (feminin)
Toute autre valeur = anomalie de format

In [5]:
# RG03 - Format Genre
valeurs_autorisees_genre = ["'M'", "'F'"]

anomalies_rg03 = df[~df["gender"].isin(valeurs_autorisees_genre)]

print(f"RG03 - Nombre d'anomalies : {len(anomalies_rg03)}")
print(f"Valeurs trouvees : {df['gender'].unique()}")

RG03 - Nombre d'anomalies : 1693
Valeurs trouvees : <StringArray>
[''M'', ''F'', ''E'', ''U'']
Length: 4, dtype: str


In [6]:
# RG03 mise a jour
valeurs_autorisees_genre = ["'M'", "'F'", "'E'"]

anomalies_rg03_technique = df[~df["gender"].isin(valeurs_autorisees_genre)]
anomalies_rg03_inconnu = df[df["gender"] == "'U'"]

print(f"RG03 - Anomalies techniques : {len(anomalies_rg03_technique)}")
print(f"RG03 - Genre inconnu : {len(anomalies_rg03_inconnu)}")

# Repartition des genres
print(f"\nRepartition des genres :")
print(df["gender"].value_counts())

RG03 - Anomalies techniques : 515
RG03 - Genre inconnu : 515

Repartition des genres :
gender
'F'    324565
'M'    268385
'E'      1178
'U'       515
Name: count, dtype: int64


In [7]:
# Analyse des transactions par genre
# On veut comprendre le comportement de chaque segment

analyse_genre = df.groupby("gender")["amount"].agg(["mean", "count", "sum"]).round(2)
analyse_genre.columns = ["montant_moyen", "nb_transactions", "montant_total"]
analyse_genre = analyse_genre.sort_values("montant_moyen", ascending=False)

print("=== Analyse des transactions par genre ===")
print(analyse_genre)

=== Analyse des transactions par genre ===
        montant_moyen  nb_transactions  montant_total
gender                                               
'F'             39.21           324565    12727181.50
'E'             36.63             1178       43147.34
'M'             36.31           268385     9744547.79
'U'             31.51              515       16227.10


## RG04 - Coherence : Montant doit etre strictement superieur a 0
Un montant de 0 ou negatif n'a pas de sens metier.
Un virement de 0€ = transaction inutile ou erreur systeme.
Un montant negatif = anomalie critique.

In [8]:
# RG04 - Coherence Montant
anomalies_rg04 = df[df["amount"] <= 0]

print(f"RG04 - Nombre d'anomalies : {len(anomalies_rg04)}")

if len(anomalies_rg04) > 0:
    print(f"\nDetails des anomalies :")
    print(anomalies_rg04[["customer", "amount", "category"]].head(10))

RG04 - Nombre d'anomalies : 52

Details des anomalies :
             customer  amount             category
40502   'C1918953803'     0.0  'es_transportation'
46361    'C782199851'     0.0  'es_transportation'
48560     'C92282564'     0.0  'es_transportation'
64613    'C612357573'     0.0  'es_transportation'
84320   'C1299474405'     0.0  'es_transportation'
87713   'C1814870538'     0.0  'es_transportation'
88889   'C1189224644'     0.0  'es_transportation'
89155   'C1598762673'     0.0  'es_transportation'
127028   'C876944738'     0.0  'es_transportation'
133555    'C73919470'     0.0  'es_transportation'


## RG05 - Unicite : Chaque transaction doit etre unique
Un doublon = une transaction enregistree deux fois dans le SI.
Consequence : un client debite deux fois pour un seul achat.

In [ ]:
# RG05 - Unicite
# On cherche les lignes dupliquees
# duplicated() retourne True si la ligne est un doublon

doublons = df[df.duplicated()]

print(f"RG05 - Nombre de doublons : {len(doublons)}")

if len(doublons) > 0:
    print(f"\nExemple de doublons :")
    print(doublons.head())